In [ ]:
# Install dependencies with FlashAttention 2 for faster inference
!pip install ninja packaging wheel
!MAX_JOBS=4 pip install -U flash-attn --no-build-isolation

# Install TTS and audio packages
!pip install -q qwen-tts soundfile gradio
!apt-get install -qq sox libsox-fmt-all

print("✅ All dependencies installed successfully!")
print("   - FlashAttention 2 (faster generation)")
print("   - Qwen-TTS package")
print("   - SoundFile (audio processing)")
print("   - Gradio (web interface)")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 64.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
import torch
import soundfile as sf
import gradio as gr
import numpy as np
from qwen_tts import Qwen3TTSModel

print("Loading Qwen3-TTS Base model with FlashAttention 2...")
print("(This takes 3-5 minutes on first run)")

model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    device_map="cuda:0",
    dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",  # Enable FlashAttention 2
)

print("✅ Model loaded successfully with FlashAttention 2!")
print("   You can now clone voices in the interface below.")


In [ ]:
def clone_voice(new_text, language, reference_audio, ref_transcript):
    """
    Clone a voice and generate speech in that cloned voice.

    Args:
        new_text: The text you want the cloned voice to speak
        language: Target language for synthesis
        reference_audio: Audio file path (3-10 seconds of clear speech)
        ref_transcript: Exact transcript of what's said in reference audio
    """
    try:
        if reference_audio is None:
            return None, "❌ Please upload a reference audio file (3-10 seconds recommended)"

        if not ref_transcript or ref_transcript.strip() == "":
            return None, "❌ Please provide the transcript of your reference audio"

        # Generate voice clone using official Qwen3-TTS API
        wavs, sr = model.generate_voice_clone(
            text=new_text,
            language=language,
            ref_audio=reference_audio,
            ref_text=ref_transcript,
        )

        # Handle output format
        if isinstance(wavs, (list, tuple)):
            audio_data = np.array(wavs[0])
        else:
            audio_data = np.array(wavs)

        return (int(sr), audio_data), f"✅ Voice cloned successfully! Sample rate: {sr}Hz"

    except Exception as e:
        import traceback
        return None, f"❌ Error: {str(e)}\n\n{traceback.format_exc()}"

# Supported languages (10 major languages)
languages = [
    "Chinese", "English", "Japanese", "Korean",
    "German", "French", "Russian", "Portuguese",
    "Spanish", "Italian"
]

# Create Gradio interface
interface = gr.Interface(
    fn=clone_voice,
    inputs=[
        gr.Textbox(
            label="📝 NEW Text (what you want the cloned voice to say)",
            placeholder="Enter the text you want to speak in the cloned voice...",
            lines=4,
            value="Hello! This is my cloned voice speaking new words."
        ),
        gr.Dropdown(
            choices=languages,
            value="English",
            label="🌐 Language"
        ),
        gr.Audio(
            label="🎤 Reference Audio (3-10 seconds of clear speech)",
            type="filepath",
            sources=["upload", "microphone"]
        ),
        gr.Textbox(
            label="📄 Reference Audio Transcript",
            placeholder="Type EXACTLY what is spoken in the reference audio above...",
            lines=3,
            value=""
        )
    ],
    outputs=[
        gr.Audio(label="🔊 Cloned Voice Output", type="numpy"),
        gr.Textbox(label="📊 Status", lines=2)
    ],
    title="🎙️ Qwen3-TTS Voice Cloning",
    description="""
    **Clone any voice from just 3 seconds of audio!**

    **How to use:**
    1. Upload a 3-10 second audio clip of the voice you want to clone
    2. Type exactly what is said in that audio (the transcript)
    3. Enter the new text you want this voice to speak
    4. Click Submit and wait for your cloned voice!

    **Tips for best results:**
    - Use clear audio without background noise
    - Make sure your transcript matches the audio exactly
    - Longer reference audio (10-20 seconds) gives better results
    - Works in 10 languages!
    """,
    examples=[
        [
            "Welcome to my AI voice cloning demo!",
            "English",
            None,
            ""
        ],
        [
            "This technology is amazing!",
            "English",
            None,
            ""
        ]
    ]
)

# Launch the interface
print("🚀 Launching Gradio interface...")
interface.launch(share=True, debug=False)
